# AIS SFP Distance Prediction Training

Train an image-based plug-tip offset predictor from the locally collected SFP dataset.
Data is expected under `ws_aic/data/distance_prediction/SFP`, and checkpoints are saved under `ws_aic/model/ais_distance_prediction`.


In [1]:
from pathlib import Path
import sys

PROJECT_DIR = Path('/home/whyz/aic_sejong/ws_aic/src/ais/ais_distance_prediction')
WS_ROOT = PROJECT_DIR.parents[2]
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

PROJECT_DIR, WS_ROOT


(PosixPath('/home/whyz/aic_sejong/ws_aic/src/ais/ais_distance_prediction'),
 PosixPath('/home/whyz/aic_sejong/ws_aic'))

In [2]:
from collections import Counter

import torch
from torch.utils.data import DataLoader
from torchvision import transforms

from model import (
    DEFAULT_WEIGHT_ROOT,
    VisionOffsetDataset,
    build_resnet_position_model,
    compute_target_stats,
    evaluate,
    fit,
    load_samples,
    seed_everything,
)

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device


/home/whyz/aic_sejong/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device(type='cuda')

In [3]:
DATASET_ROOT = WS_ROOT / 'data' / 'distance_prediction' / 'SFP'
WEIGHT_ROOT = DEFAULT_WEIGHT_ROOT

CAMERAS = ('left', 'center', 'right')
TARGET_KEYS = ('x_mm', 'y_mm', 'z_mm')
IMAGE_SIZE = (256, 288)
BATCH_SIZE = 8
NUM_WORKERS = 8
EPOCHS = 80
LR = 3e-4
WEIGHT_DECAY = 1e-4
BACKBONE = 'resnet50'
PRETRAINED = True
AGGREGATION = 'concat'
NUM_PORT_HEADS = 2
EXPAND_ALL_PORTS = False
VAL_EVERY_N = 5
RUN_NAME = f"sfp_distance_{BACKBONE}_{'_'.join(CAMERAS)}_{AGGREGATION}"

DATASET_ROOT, WEIGHT_ROOT / RUN_NAME


(PosixPath('/home/whyz/aic_sejong/ws_aic/data/distance_prediction/SFP'),
 PosixPath('/home/whyz/aic_sejong/ws_aic/model/ais_distance_prediction/sfp_distance_resnet50_left_center_right_concat'))

In [4]:
def split_samples_by_grid_holdout(samples, *, every_n=5):
    train, val = [], []
    for sample in samples:
        row = dict(sample)
        step_index = int(row.get('step_index', 0))
        if step_index % every_n == 0:
            val.append(row)
        else:
            train.append(row)
    return train, val, []


if not (DATASET_ROOT / 'samples.jsonl').is_file():
    raise FileNotFoundError(f'Missing SFP samples.jsonl: {DATASET_ROOT / "samples.jsonl"}')

samples = load_samples(DATASET_ROOT)
missing_images = [
    DATASET_ROOT / sample['images'][camera]
    for sample in samples
    for camera in CAMERAS
    if not (DATASET_ROOT / sample['images'][camera]).is_file()
]
if missing_images:
    preview = '\n'.join(str(path) for path in missing_images[:5])
    raise FileNotFoundError(f'{len(missing_images)} image files are missing. First missing files:\n{preview}')

train_samples, val_samples, test_samples = split_samples_by_grid_holdout(
    samples,
    every_n=VAL_EVERY_N,
)
target_stats = compute_target_stats(
    train_samples,
    TARGET_KEYS,
    expand_all_ports=EXPAND_ALL_PORTS,
)

print(f'dataset_root: {DATASET_ROOT}')
print(f'raw samples: total={len(samples)} train={len(train_samples)} val={len(val_samples)} test={len(test_samples)}')
print(f'effective rows: train={len(train_samples) * (NUM_PORT_HEADS if EXPAND_ALL_PORTS else 1)} '
      f'val={len(val_samples) * (NUM_PORT_HEADS if EXPAND_ALL_PORTS else 1)}')
print(f'images checked: {len(samples) * len(CAMERAS)}')
print('ports:', dict(Counter(sample.get('port_name', 'unknown') for sample in samples)))
print('label frame example:', samples[0]['label']['frame'])
print('target mean:', target_stats['mean'].tolist())
print('target std :', target_stats['std'].tolist())


dataset_root: /home/whyz/aic_sejong/ws_aic/data/distance_prediction/SFP
raw samples: total=768 train=614 val=154 test=0
effective rows: train=614 val=154
images checked: 2304
ports: {'sfp_port_0': 384, 'sfp_port_1': 384}
label frame example: task_board/nic_card_mount_0/sfp_port_0_link_entrance
target mean: [-0.16599492728710175, -0.205885112285614, -19.548357009887695]
target std : [1.0524181127548218, 2.2005114555358887, 3.375432014465332]


In [5]:
train_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

eval_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

In [6]:
train_dataset = VisionOffsetDataset(
    DATASET_ROOT,
    samples=train_samples,
    cameras=CAMERAS,
    target_keys=TARGET_KEYS,
    transform=train_transform,
    target_mean=target_stats['mean'],
    target_std=target_stats['std'],
    expand_all_ports=EXPAND_ALL_PORTS,
)
val_dataset = VisionOffsetDataset(
    DATASET_ROOT,
    samples=val_samples,
    cameras=CAMERAS,
    target_keys=TARGET_KEYS,
    transform=eval_transform,
    target_mean=target_stats['mean'],
    target_std=target_stats['std'],
    expand_all_ports=EXPAND_ALL_PORTS,
)

pin_memory = device.type == 'cuda'
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    drop_last=False,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    drop_last=False,
)

batch = next(iter(train_loader))
batch['image'].shape, batch['target'].shape, batch['port_id'][:8], batch['raw_target'][:3]


(torch.Size([8, 3, 3, 256, 288]),
 torch.Size([8, 3]),
 tensor([0, 1, 1, 0, 0, 0, 0, 1]),
 tensor([[  1.1042,   0.3662, -14.0449],
         [  0.3404,   2.3857, -14.1151],
         [ -1.6076,  -0.6291, -21.1239]]))

In [7]:
model = build_resnet_position_model(
    backbone_name=BACKBONE,
    pretrained=PRETRAINED,
    output_dim=len(TARGET_KEYS),
    hidden_dim=256,
    dropout=0.1,
    aggregation=AGGREGATION,
    num_views=len(CAMERAS),
    num_port_heads=NUM_PORT_HEADS,
)
model.to(device)

optimizer = torch.optim.AdamW(
    (p for p in model.parameters() if p.requires_grad),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3,
)

sum(p.numel() for p in model.parameters() if p.requires_grad)


26720838

In [8]:
config = {
    'dataset_root': str(DATASET_ROOT),
    'cameras': CAMERAS,
    'target_keys': TARGET_KEYS,
    'target_mean': target_stats['mean'].tolist(),
    'target_std': target_stats['std'].tolist(),
    'image_size': IMAGE_SIZE,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'lr': LR,
    'weight_decay': WEIGHT_DECAY,
    'backbone': BACKBONE,
    'pretrained': PRETRAINED,
    'aggregation': AGGREGATION,
    'num_views': len(CAMERAS),
    'num_port_heads': NUM_PORT_HEADS,
    'expand_all_ports': EXPAND_ALL_PORTS,
    'label_frame_mode': samples[0]['label'].get('frame_mode'),
    'label_coordinate': samples[0]['label'].get('coordinate'),
    'split': {'name': 'grid_holdout', 'val_every_n': VAL_EVERY_N},
}

history = fit(
    model,
    train_loader,
    val_loader,
    optimizer,
    device,
    epochs=EPOCHS,
    weight_dir=WEIGHT_ROOT,
    run_name=RUN_NAME,
    scheduler=scheduler,
    target_mean=target_stats['mean'],
    target_std=target_stats['std'],
    config=config,
)
print(f'best checkpoint: {WEIGHT_ROOT / RUN_NAME / "best.pt"}')
history[-1]


epochs:   1%|▏         | 1/80 [00:42<55:34, 42.21s/it, train_loss=0.2311, val_euclidean=1.789, val_loss=0.1638]

epoch=001 train_loss=0.2311 val_loss=0.1638 val_mae=0.913 val_euclidean=1.789


epochs:   2%|▎         | 2/80 [00:52<30:49, 23.71s/it, train_loss=0.0701, val_euclidean=1.041, val_loss=0.0308]

epoch=002 train_loss=0.0701 val_loss=0.0308 val_mae=0.473 val_euclidean=1.041


epochs:   4%|▍         | 3/80 [01:03<22:49, 17.79s/it, train_loss=0.0448, val_euclidean=0.700, val_loss=0.0156]

...

epochs:  99%|█████████▉| 79/80 [14:33<00:10, 10.67s/it, train_loss=0.0066, val_euclidean=0.112, val_loss=0.0004]

epoch=079 train_loss=0.0066 val_loss=0.0004 val_mae=0.053 val_euclidean=0.112


epochs: 100%|██████████| 80/80 [14:43<00:00, 11.05s/it, train_loss=0.0067, val_euclidean=0.116, val_loss=0.0005]

epoch=080 train_loss=0.0067 val_loss=0.0005 val_mae=0.055 val_euclidean=0.116
best checkpoint: /home/whyz/aic_sejong/ws_aic/model/ais_distance_prediction/sfp_distance_resnet50_left_center_right_concat/best.pt


{'epoch': 80.0,
 'train_loss': 0.0067251004302928815,
 'val_loss': 0.000489985566531828,
 'val_mae': 0.054668448865413666,
 'val_rmse': 0.081244096159935,
 'val_euclidean': 0.11614071577787399}

In [9]:
val_metrics = evaluate(
    model,
    val_loader,
    device,
    target_mean=target_stats['mean'],
    target_std=target_stats['std'],
)
val_metrics

{'loss': 0.000489985566531828,
 'mae': 0.054668448865413666,
 'rmse': 0.081244096159935,
 'euclidean': 0.11614071577787399}

In [10]:
model.eval()
batch = next(iter(val_loader))
with torch.inference_mode():
    pred = model(batch['image'].to(device), batch['port_id'].to(device)).cpu()
pred_mm = pred * target_stats['std'] + target_stats['mean']
preview = torch.cat([pred_mm[:8], batch['raw_target'][:8]], dim=1)
print('columns: pred_x pred_y pred_z target_x target_y target_z')
display(preview)

correct_count = 0
total_count = 0
before_distances = []
after_distances = []

with torch.inference_mode():
    for eval_batch in val_loader:
        pred = model(eval_batch['image'].to(device), eval_batch['port_id'].to(device)).cpu()
        pred_mm = pred * target_stats['std'] + target_stats['mean']
        target_mm = eval_batch['raw_target']

        before_distance = torch.linalg.vector_norm(target_mm, dim=1)
        after_distance = torch.linalg.vector_norm(target_mm - pred_mm, dim=1)
        improved = after_distance < before_distance

        correct_count += int(improved.sum())
        total_count += int(improved.numel())
        before_distances.append(before_distance)
        after_distances.append(after_distance)

before_distance_mm = torch.cat(before_distances)
after_distance_mm = torch.cat(after_distances)
improvement_metrics = {
    'criterion': 'after_distance_mm < before_distance_mm',
    'accuracy': correct_count / max(total_count, 1),
    'correct': correct_count,
    'total': total_count,
    'before_distance_mean_mm': float(before_distance_mm.mean()),
    'after_distance_mean_mm': float(after_distance_mm.mean()),
    'distance_improvement_mean_mm': float((before_distance_mm - after_distance_mm).mean()),
}
improvement_metrics


columns: pred_x pred_y pred_z target_x target_y target_z


tensor([[ -1.5234,  -3.5569, -14.5144,  -1.6394,  -3.6407, -14.0815],
        [ -1.6311,  -3.5958, -19.0500,  -1.6312,  -3.6579, -19.0862],
        [ -1.5995,  -3.5588, -24.1175,  -1.5583,  -3.5894, -24.0825],
        [ -1.6153,  -2.5697, -16.9571,  -1.6706,  -2.6444, -17.0917],
        [ -1.6198,  -2.6304, -21.9708,  -1.6195,  -2.6530, -22.0699],
        [ -1.6314,  -1.5902, -15.1625,  -1.6549,  -1.6468, -15.0422],
        [ -1.6168,  -1.6527, -19.9416,  -1.6312,  -1.6182, -20.0610],
        [ -1.5963,  -1.6172, -24.2905,  -1.6004,  -1.6602, -24.5555]])

{'criterion': 'after_distance_mm < before_distance_mm',
 'accuracy': 1.0,
 'correct': 154,
 'total': 154,
 'before_distance_mean_mm': 19.685129165649414,
 'after_distance_mean_mm': 0.11614072322845459,
 'distance_improvement_mean_mm': 19.568984985351562}